# Track 3: a fresh Devanagari probe, base vs the shipped adapter

30 tasks the model has never seen, all six hard families, five each, every sentence in Devanagari
with Latin names and ISO codes unchanged. New seed (20261115), disjoint from the earlier
Devanagari-30 and from every other split in the repo, checked by exact task text.

| # | run folder | model | time on one T4 |
|---|---|---|---:|
| 1 | `sft4_hindi30` | shipped `sft4` adapter | ~25 min |
| 2 | `base_hindi30` | untuned `CohereLabs/tiny-aya-global` | ~45 min |

The adapter runs first so a session that ends early still banks the headline number.

Why both: the published Devanagari row (26/30 AuditKIT, 25/30 repo scorer) has no base control, and
it reuses a split that was built before the 257-task holdout existed. This run answers "does the
recipe survive a language script it was not tuned on" with a split nobody has scored before, and
pairs every task against the untuned model, so the delta is per-task, not two headline numbers.

The split is embedded in this notebook (base64, SHA-256 checked), so it runs against the pinned
commit without waiting for a repo push. Each cell uploads its run to `primce/lexsi-runs` as soon as
it finishes, so a quota stop keeps whatever is already earned. One GPU only: Kaggle bills a T4x2
machine as two GPU-hours per hour.

In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1

## Setup

The clone is pinned to `b254024` - the same commit every published number comes from. The
offline checks are CPU-only and pin every frozen split by SHA-256, so a broken clone fails here
instead of producing a number that cannot be compared with the published rows.

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git checkout -q b2540242c7ad902af6792bab6cff39116b46d578
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py

## The split, written from the notebook

The 30 rows travel inside this file, so the probe is reproducible without publishing the split
first. The SHA-256 below is the value frozen in `data/hindi30_eval.sha256`; if it does not match,
stop - the numbers would not be comparable.

In [ ]:
import base64
import hashlib
import json
from pathlib import Path

B64 = ''.join([
    "...",
    "bHkiOiAiZW1wdHlfYmFsYW5jZSJ9Cg==",
])

raw = base64.b64decode(B64)
path = Path("lexsi-agentic-hinglish/data/tasks_hindi30_eval.jsonl")
path.write_bytes(raw)
digest = hashlib.sha256(raw).hexdigest()
assert digest == "2b3179acbc0600fd0e474adaf43e177bb84d9895575243e229e2759aa58acd43", digest
rows = [json.loads(line) for line in raw.decode("utf-8").splitlines()]
families = {}
for row in rows:
    families[row["family"]] = families.get(row["family"], 0) + 1
print(len(rows), "tasks, sha256", digest)
print(sorted(families.items()))
print("ids", rows[0]["id"], "..", rows[-1]["id"])

## The shipped adapter

`sft4` is the seed-42 recipe (hard-30 29/30, hard-90 84/90, hard-257 235/257). Only `sft4_adapter/`
is downloaded; the checkpoints inside the repo are not needed for evaluation.

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
python - <<'PY'
from huggingface_hub import snapshot_download

path = snapshot_download(
    "primce/tiny-aya-hinglish-lora",
    allow_patterns=["sft4_adapter/adapter_config.json",
                   "sft4_adapter/adapter_model.safetensors",
                   "sft4_adapter/tokenizer.json",
                   "sft4_adapter/tokenizer_config.json"],
    local_dir="runs/_adapter_dl",
    token=__import__("os").environ["HF_TOKEN"],
)
print("adapter ->", path)
PY
test -f runs/_adapter_dl/sft4_adapter/adapter_config.json && echo "adapter present"

## 1. The shipped adapter

Same 30 tasks, same prompt, canonical tool schema. The only variable is the model.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_hindi30_eval.jsonl \
  --no-4bit \
  --out runs/sft4_hindi30 || echo "SFT4 RUN FAILED"

python scripts/job_upload.py --run sft4_hindi30 || echo "SFT4 UPLOAD FAILED"

## 2. The untuned base (control)

No adapter. On the English-script hard-30 this model scores 1/30; the question is whether
Devanagari makes the task easier, harder, or just differently broken.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_hindi30_eval.jsonl \
  --no-4bit \
  --out runs/base_hindi30 || echo "BASE RUN FAILED"

python scripts/job_upload.py --run base_hindi30 || echo "BASE UPLOAD FAILED"

## Collect

Prints both summaries and the paired table: per task, base outcome vs adapter outcome, so the
McNemar cells are visible without re-reading trajectories.

In [ ]:
import json
from pathlib import Path

root = Path("lexsi-agentic-hinglish/runs")
keys = ("n", "success", "tac", "answer_ok", "strict_rate", "calls", "parse_failures", "seconds")

for name in ("base_hindi30", "sft4_hindi30"):
    path = root / name / "summary.json"
    if not path.exists():
        print(name, "MISSING")
        continue
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(name, {key: summary[key] for key in keys if key in summary})
    for family, row in (summary.get("families") or {}).items():
        print(f"   {family}: n={row.get('n')} success={row.get('success')}")

def outcomes(name):
    out = {}
    for line in (root / name / "results.jsonl").read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        out[row["task_id"]] = bool(row["success"])
    return out

base_path = root / "base_hindi30" / "results.jsonl"
sft_path = root / "sft4_hindi30" / "results.jsonl"
if base_path.exists() and sft_path.exists():
    base, sft = outcomes("base_hindi30"), outcomes("sft4_hindi30")
    ids = sorted(base)
    fixed = sum(1 for i in ids if sft[i] and not base[i])
    broken = sum(1 for i in ids if base[i] and not sft[i])
    neither = sum(1 for i in ids if not base[i] and not sft[i])
    print(f"paired over {len(ids)}: fixed={fixed} broken={broken} neither={neither}"
          f" base={sum(base.values())} sft4={sum(sft.values())}")
    print("base misses:", [i for i in ids if not base[i]])
    print("sft4 misses:", [i for i in ids if not sft[i]])